<a href="https://colab.research.google.com/github/ShinAsakawa/ShinAsakawa.github.io/blob/master/2026notebooks/2026_0406Consistency_table_NTT_psylex71.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# NTT日本語語彙特性データ 単語頻度データ `psylex71.txt` のアップロード
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

In [ ]:
from typing import List, Tuple
import numpy as np
import pandas as pd
from collections import Counter
import re

In [ ]:
# 日本語文字コードの正規化 normalize について
try:
    import jaconv
except ImportError:
    !pip install jaconv
    import jaconv

str = 'ﾃｨﾛ･フィナ〜レ'
print(str, jaconv.normalize(str))

In [ ]:
# -----------------------------
# Mora segmentation (simple)
# -----------------------------
SMALL_KANA = set(list("ゃゅょぁぃぅぇぉャュョァィゥェォ"))
LONG_VOWEL = "ー"
SOKUON = set(["っ", "ッ"])
HATSUON = set(["ん", "ン"])

def kana_to_mora(reading: str):
    """
    Convert kana string to a list of morae.
    Examples:
      がっこう -> ['ガ', 'ッ', 'コ', 'ー']
      きょう   -> ['キョ', 'ー']
    """
    morae = []
    i = 0
    while i < len(reading):
        ch = reading[i]

        # sokuon or hatsuon
        if ch in SOKUON or ch in HATSUON:
            morae.append(ch)
            i += 1
            continue

        # base kana + small kana
        if i + 1 < len(reading) and reading[i + 1] in SMALL_KANA:
            morae.append(ch + reading[i + 1])
            i += 2
            continue

        # long vowel marker
        if ch == LONG_VOWEL:
            morae.append(ch)
            i += 1
            continue

        morae.append(ch)
        i += 1

    return morae

def kata_to_hira(s: str) -> str:
    """Convert katakana to hiragana"""
    res = []
    for ch in s:
        o = ord(ch)
        # Katakana Unicode range
        if 0x30A1 <= o <= 0x30F6:
            res.append(chr(o - 0x60))
        else:
            res.append(ch)
    return "".join(res)

In [ ]:
import math
from collections import Counter, defaultdict

def build_kanji_consistency(data):
    """
    Returns:
      cons[c] in [0,1] : max-prob of (some reading signature) given kanji c
    Here we use first mora as a cheap signature; you can replace with full mora seq,
    or mora bigram, or onset-vowel patterns.
    """
    dist = defaultdict(Counter)
    for w, r in data:
        morae = kana_to_mora(r)
        sig = morae[0] if len(morae) > 0 else "<nil>"
        for c in w:  # 2 chars
            dist[c][sig] += 1

    cons = {}
    for c, cnt in dist.items():
        total = sum(cnt.values())
        cons[c] = (max(cnt.values()) / total) if total > 0 else 0.0
    return cons

def word_consistency(word2: str) -> float:
    # 単語の一貫性＝2文字の平均（他にも min, product などあり）
    return 0.5 * (KANJI_CONS.get(word2[0], 0.0) + KANJI_CONS.get(word2[1], 0.0))

In [ ]:
# import os
# HOME = os.environ['HOME']
# psylex71_path = os.path.join(HOME, 'study/2017_2009AmanoKondo_NTTKanjiData/psylex71utf8.txt')
# HOME = os.environ['HOME']
psylex71_path = 'psylex71.txt'

def load_psylex71_data(psylex71_path:str):
    kata_chars = 'ァアィイゥウェエォオカガキギクグケゲコゴサザシジスズセゼソゾタダチヂッツヅテデトドナニヌネノハバパヒビピフブプヘベペホボポマミムメモャヤュユョヨラリルレロヮワヰヱヲンヴヵ゛゛゜ー—'
    kata_firts = 'アイウエオカガキギクグケゲコゴサザシジスズセゼソゾタダチヂッツヅテデトドナニヌネノハバパヒビピフブプヘベペホボポマミムメモヤユヨラリルレロワヰヱヲンヴヵ'
    KANJI_RE = re.compile(r"^[一-龯]{2}$")

    data = []
    with open(psylex71_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(' ')  # 一行を空白で分割

            word = jaconv.normalize(parts[2])  # 表記
            yomi = jaconv.normalize(parts[3])  # 読み
            pos  = parts[4]  # 品詞
            try:
                freq = int(parts[5])       # 頻度全体
            except ValueError:
                continue

            # Filter: 漢字２文字からなる単語だけを取り出す
            if not KANJI_RE.match(word):
                continue

            is_valid_yomi = False
            if yomi[0] in kata_firts:
                is_valid_yomi = True
                for y in yomi[1:]:
                    if not y in kata_chars:
                        is_valid_yomi = False
            if is_valid_yomi:
                data.append((word, yomi,freq))

    print(f"Loaded {len(data)} 2-kanji words from {psylex71_path}")
    return data

psylex_data = load_psylex71_data(psylex71_path)
print(f'len(psylex_data):{len(psylex_data)}')


# CDP+ 用（頻度は別で保持）
DATA = [(w, r) for (w, r, f) in psylex_data]

# frequency dictionary
word_freq = {w: f for (w, r, f) in psylex_data}

KANJI_CONS = build_kanji_consistency(DATA)

In [ ]:
for entry in psylex_data[-30:]:
    print(f'単語:{entry[0]}, ヨミ:{entry[1]}, 一貫性:{word_consistency(entry[0]):.3f}, 頻度:{entry[2]}')